# 02 - Aumento de Dados (Data Augmentation) - SinucaVision

Este notebook expande o nosso dataset original de imagens de sinuca criando dezenas de variações sintéticas. Ele aplica transformações geométricas (rotação, espelhamento) e fotométricas (brilho, contraste, ruído) para simular as condições de iluminação de um bar ou salão de jogos.

**Estrutura de pastas esperada:**
* Entrada: `../data/dataset_origem/`
* Saída: `../data/dataset_aumentado/`

#### Instruções para baixar o dataset  com as anotações realizadas no Roboflow

Para aumentar o dataset, foi realizado um projeto no Roboflow com o objetivo de gerar anotações manuais das posições das bolas na mesa. Após o download deve se organizar o dataset em uma pasta (em .data) nomeada de **dataset_origem** somente com dauas subpastas: **images** e **labels** (sem diferenciação entre train, test e val).

In [ ]:
%pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="RprFM1by45ZMbCtG5zJt")
project = rf.workspace("heloisas-workspace").project("detector-de-bolas-com-yolo")
version = project.version(2)
dataset = version.download("yolov8")          

#### Importação de Bibliotecas

In [ ]:
import os
import random
import glob
import cv2
import numpy as np
import albumentations as A

print("Bibliotecas importadas com sucesso!")

## Célula 1 - Configurações do Dataset
Aqui definimos onde estão as imagens originais, onde o novo dataset será salvo e quantas variações queremos gerar por imagem.

In [ ]:
PASTA_ORIGEM = os.path.join('..', 'data', 'dataset_origem')
PASTA_DESTINO = os.path.join('..', 'data', 'dataset_aumentado')

N_VARIACOES = 30           
FRACAO_VALIDACAO = 0.15    # 15% das imagens vão para a pasta 'val' (validação)

print(f"Origem configurada para: {PASTA_ORIGEM}")
print(f"Destino configurado para: {PASTA_DESTINO}")

## Célula 2 - Funções Auxiliares
Estas funções ajudam a ler e salvar as coordenadas das marcações (bounding boxes) no formato que o YOLO exige. Também incluímos funções para desenhar sombras e reflexos artificiais nas mesas de sinuca.

In [ ]:
def carregar_labels_yolo(caminho_txt):
    """Lê o arquivo YOLO e retorna listas de caixas e classes."""
    
    boxes, classes = [], []
    if not os.path.exists(caminho_txt):
        return boxes, classes
    with open(caminho_txt) as f:
        for linha in f:
            partes = linha.strip().split()
            if len(partes) != 5:
                continue
            c, cx, cy, w, h = partes
            classes.append(int(c))
            boxes.append([float(cx), float(cy), float(w), float(h)])
    return boxes, classes

def salvar_labels_yolo(caminho_txt, boxes, classes):
    """Salva as novas coordenadas geradas num arquivo .txt."""

    with open(caminho_txt, 'w') as f:
        for (cx, cy, w, h), c in zip(boxes, classes):
            f.write(f"{c} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")

def adicionar_sombra_sintetica(img, intensidade_max=0.5):
    """Desenha um polígono escuro simulando a sombra de um jogador na mesa."""

    h, w = img.shape[:2]
    overlay = img.copy()
    n_pontos = random.randint(3, 6)
    pontos = np.array([
        [random.randint(0, w), random.randint(0, h)] for _ in range(n_pontos)
    ], dtype=np.int32)
    cv2.fillPoly(overlay, [pontos], (0, 0, 0))
    alpha = random.uniform(0.15, intensidade_max)
    return cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0)

def adicionar_reflexo_sintetico(img, intensidade_max=0.4):
    """Adiciona uma mancha clara para simular o reflexo de uma luminária pendente."""

    h, w = img.shape[:2]
    overlay = img.copy()
    cx, cy = random.randint(0, w), random.randint(0, h)
    raio = random.randint(int(min(h, w) * 0.05), int(min(h, w) * 0.2))
    cv2.circle(overlay, (cx, cy), raio, (255, 255, 255), -1)
    overlay = cv2.GaussianBlur(overlay, (51, 51), 0)
    alpha = random.uniform(0.1, intensidade_max)
    return cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0)

## Célula 3 - Pipeline de Transformação
Aqui usamos a biblioteca **Albumentations**. Sua função é recalcular automaticamente as coordenadas das caixas (bounding boxes) para que as marcações acompanhem as bolas.

In [ ]:
def construir_transform():
    """Define as regras de distorção da imagem."""

    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.Rotate(limit=8, border_mode=cv2.BORDER_REPLICATE, p=0.6),
        A.RandomBrightnessContrast(brightness_limit=0.35, contrast_limit=0.35, p=0.9),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=30, val_shift_limit=30, p=0.7),
        A.GaussNoise(var_limit=(5.0, 30.0), p=0.4),
        A.MotionBlur(blur_limit=3, p=0.2),
        A.CLAHE(p=0.2),
        A.RandomShadow(shadow_roi=(0, 0, 1, 1), num_shadows_limit=(1, 2), p=0.3),
    ], bbox_params=A.BboxParams(format='yolo', label_fields=['classes']))

def gerar_variacoes(caminho_img, caminho_txt, pasta_img_out, pasta_lbl_out, prefixo, n):
    """Aplica o pipeline N vezes em uma única imagem."""
    
    img = cv2.imread(caminho_img)
    if img is None:
        print(f'  [ERRO] não foi possível ler {caminho_img}')
        return 0

    boxes, classes = carregar_labels_yolo(caminho_txt)
    if not boxes:
        print(f'  [AVISO] {caminho_txt} sem anotações — pulando')
        return 0

    transform = construir_transform()
    gerados = 0

    for i in range(n):
        try:
            aug = transform(image=img, bboxes=boxes, classes=classes)
        except Exception:
            continue # Ignora se a transformação jogar a caixa para fora da imagem

        img_aug = aug['image']
        boxes_aug = aug['bboxes']
        classes_aug = aug['classes']

        if not boxes_aug:
            continue 

        # Efeitos manuais específicos para o cenário de sinuca
        if random.random() < 0.5:
            img_aug = adicionar_sombra_sintetica(img_aug)
        if random.random() < 0.3:
            img_aug = adicionar_reflexo_sintetico(img_aug)

        # Salva o resultado
        nome_saida = f'{prefixo}_aug{i:03d}'
        cv2.imwrite(os.path.join(pasta_img_out, nome_saida + '.jpg'), img_aug)
        salvar_labels_yolo(os.path.join(pasta_lbl_out, nome_saida + '.txt'), boxes_aug, classes_aug)
        gerados += 1

    return gerados

## Célula 4 - Execução Principal
Esta célula lê a pasta de origem, cria a estrutura de pastas do destino (`train/` e `val/`), distribui as imagens originais e gera as dezenas de variações estipuladas na configuração. No fim, ela cria automaticamente o arquivo `data.yaml` atualizado para as nossas 4 classes.

In [ ]:
imagens_src = sorted(
    glob.glob(os.path.join(PASTA_ORIGEM, 'images', '*.jpg')) +
    glob.glob(os.path.join(PASTA_ORIGEM, 'images', '*.JPG')) +
    glob.glob(os.path.join(PASTA_ORIGEM, 'images', '*.png'))
)

print(f'Imagens originais encontradas: {len(imagens_src)}')

if len(imagens_src) > 0:
    # Criar estrutura de pastas do YOLO
    for split in ['train', 'val']:
        os.makedirs(os.path.join(PASTA_DESTINO, 'images', split), exist_ok=True)
        os.makedirs(os.path.join(PASTA_DESTINO, 'labels', split), exist_ok=True)

    # Separar as imagens entre treino e validação
    n_val = max(1, round(len(imagens_src) * FRACAO_VALIDACAO))
    indices_val = set(random.sample(range(len(imagens_src)), k=min(n_val, len(imagens_src))))
    if len(imagens_src) > 1 and len(indices_val) == len(imagens_src):
        indices_val = set(list(indices_val)[:-1]) # Garante que treino não fique vazio
        
    print(f'Split: {len(imagens_src) - len(indices_val)} imagens em train, {len(indices_val)} em val\n')

    # Gerar o Dataset
    total_gerado = 0
    for idx, caminho_img in enumerate(imagens_src):
        nome_base = os.path.splitext(os.path.basename(caminho_img))[0]
        caminho_txt = os.path.join(PASTA_ORIGEM, 'labels', nome_base + '.txt')

        split = 'val' if idx in indices_val else 'train'
        pasta_img_out = os.path.join(PASTA_DESTINO, 'images', split)
        pasta_lbl_out = os.path.join(PASTA_DESTINO, 'labels', split)

        # Salva a imagem original também no novo dataset
        img_original = cv2.imread(caminho_img)
        cv2.imwrite(os.path.join(pasta_img_out, nome_base + '_orig.jpg'), img_original)
        boxes, classes = carregar_labels_yolo(caminho_txt)
        salvar_labels_yolo(os.path.join(pasta_lbl_out, nome_base + '_orig.txt'), boxes, classes)

        # Gera as variações sintéticas
        gerados = gerar_variacoes(caminho_img, caminho_txt, pasta_img_out, pasta_lbl_out, prefixo=nome_base, n=N_VARIACOES)
        total_gerado += gerados
        
        print(f'{nome_base}: {gerados} variações geradas → pasta {split}/')

    print(f'\nConcluído! Total de imagens no dataset aumentado: {total_gerado + len(imagens_src)}')
    
    # Criar o data.yaml final
    yaml_path = os.path.join(PASTA_DESTINO, 'data.yaml')
    with open(yaml_path, 'w') as f:
        f.write("train: images/train\n")
        f.write("val: images/val\n")
        f.write("nc: 4\n")
        f.write("names: ['branca', 'lisa', 'listrada', 'preta']\n")
    print(f'Arquivo de configuração salvo em: {yaml_path}')

else:
    print("Nenhuma imagem encontrada. Verifique se o caminho PASTA_ORIGEM está correto.")